# 경계 후보 추출 조합 실험

* `01_text_boundary_test.ipynb`의 후보 탐지 파이프라인(청킹 → pooled 임베딩 → depth score → 후보 선택)을 **글자 수 기준**으로 바꾼 뒤, 아래 조합마다 경계 후보 추출.
* 이후 **GoldenSet(`dataset/GoldenSet/hj/`, 회의 20개)의 정답 경계가 후보에 얼마나 포함되는지**로 가장 좋은 조합 찾는 실험


| 실험 요소 | 값 |
|---|---|
| `chunk_size` (자) | 200, 300, 400, 500 |
| `chunk_overlap` (자) | 50, 100 |
| 임베딩 모델 | `jinaai/jina-embeddings-v5-text-small`, `alphabrothers/alpha-sts-v0`, `BAAI/bge-m3` |
| 후보 선택 방식 | `threshold`: depth > 0 이고 depth ≥ 평균 + K·표준편차 / `ratio`: depth 상위 N% |

→ 조합 48개 = 청크 4 × 오버랩 2 × 모델 3 × 선택 방식 2

#### [평가 기준: 정답 경계 포함률 (재현율)]
- **포함:** 정답 경계 위치에서 ±`MATCH_TOLERANCE`발화 안에 후보가 하나라도 있으면 그 정답 경계는 후보에 포함된 것으로 봄.
- **포함률:** 회의 20개의 정답 경계를 모두 모아, 포함된 정답 경계의 비율을 계산.
- **허용 오차를 두는 이유:** 후보 위치는 청크 경계 단위로 정해지므로 발화 하나 단위로 정확히 맞추기 어려움. 다음 단계(LLM 검증)에서 앞뒤 문맥을 보고 판단하므로 경계 근처면 충분. 참고용으로 허용 오차 0(완전 일치)과 ±3도 함께 보여줌.
- **후보 수도 함께 확인:** 후보를 많이 뽑을수록 포함률은 쉽게 올라감. 그래서 순위는 **포함률이 높은 순 → 같으면 후보 수가 적은 순**으로 매기고, 후보 수와 후보 비율도 함께 표시. 후보가 많을수록 다음 단계 LLM 검증 비용이 늘어남.
- LLM 검증과 `ex_guide.md`의 평가지표는 여기서 찾은 최적 조합으로 다음 단계에서 진행.

#### * 노트북과 달라진 점
>- 모든 길이 기준(청크, 오버랩, 맞장구 제외)이 **토큰 → 글자 수(`len`, 공백·줄바꿈 포함)**으로 변경.
>- 임베딩은 **pooled 방식만** 사용.
>  > 발화마다 한 번만 임베딩해 두면, 청크 크기·오버랩이 바뀌어도 모델을 다시 돌릴 필요 X.
>- **정답 위치 변환:** GoldenSet의 `boundary_positions`는 **1부터 세는** 새 구간 첫 발화 번호임. 노트북 내부는 0부터 세므로 1을 빼서 씀.

## 0. 실행 환경

- 프로젝트 루트(`remeet/`)에서 `uv sync`가 되어 있어야 함.
- **jina v5는 `peft` 패키지가 필요.** 없으면 아래 셀의 주석을 풀고 한 번 실행한 뒤 커널을 재시작할 것.
- **라이선스 주의:** jina v5는 **CC BY-NC 4.0(비상업용)**이므로, 실험용으로는 괜찮지만 서비스에 쓰려면 별도 확인이 필요.
- **처음 실행할 때 시간이 걸림.**
  - 모델 다운로드: 합계 약 5GB
  - 임베딩: CPU 기준 모델당 수 분
  - 임베딩은 **문장 단위로 캐시**함. 같은 문장은 회의가 달라도 한 번만 계산하고, 두 번째 실행부터는 저장된 값을 바로 읽음.

In [ ]:
#!uv pip install "peft>=0.15.2"
# uv add로 추가하고 이후에 toml에서 지울 것

In [ ]:
# --------------------------------------------------------------------------------------------------
# 0-1. 실험 설정
# 경로/파라미터는 해당 셀에서만 수정
# --------------------------------------------------------------------------------------------------
from pathlib import Path

# ── 경로 ──────────────────────────────────────────────
BASE_DIR = Path.cwd().parent                                        # playground/chaewon (src/에서 실행 가정)
TRANSCRIPT_DIR = BASE_DIR.parent / "hyungjin" / "test" / "input"    # 회의별 전사문 <회의ID>.txt
GOLD_DIR = BASE_DIR / "dataset" / "GoldenSet" / "hj"                # 회의별 정답 경계 <회의ID>.json
EXP_DIR = BASE_DIR / "outputs" / "experiment"                       # 캐시·결과 저장 폴더 (.gitignore 대상)
EMBED_CACHE_DIR = EXP_DIR / "embeddings"                            # 모델별 문장 임베딩 캐시 (texts.json + vectors.npy)

MEETING_IDS = None              # None: GoldenSet의 모든 회의 / 예: ["TS3003b", "ES2004a"]로 일부만
HEADER_LINES = 3                # 전사문 상단 메타데이터 줄 수 (회의 대주제 / 참여자 / 일시)
UTTERANCE_SEPARATOR = "\n"      # 청크 안에서 발화를 이어 붙일 구분자

# ── 실험 조합 ─────────────────────────────────────────
CHUNK_SIZES = [200, 300, 400, 500]      # 청크당 최대 글자 수
OVERLAP_SIZES = [50, 100]               # 인접 청크가 공유할 최대 글자 수
EMBEDDING_MODELS = {
    # 이름: (Hugging Face 모델 ID, 로딩 옵션, encode 옵션)
    "jina-v5":    ("jinaai/jina-embeddings-v5-text-small", {"trust_remote_code": True}, {"task": "text-matching"}),
    "alpha-sts":  ("alphabrothers/alpha-sts-v0", {}, {}),
    "bge-m3":     ("BAAI/bge-m3", {}, {}),
}
SELECTION_METHODS = {
    "threshold": {"std_k": 0.0},        # depth > 0 이고 depth ≥ 평균 + std_k × 표준편차
    "ratio":     {"ratio": 0.30},       # depth 상위 30% 청크 경계
}

# ── 고정 파라미터 (01 노트북 기본값을 글자 수로 변환) ───────────
FILLER_MAX_CHARS = 4            # 이 글자 수 이하 발화(네., 음., 그러니까, 좋아요. 등)는 pooled 평균에서 제외
BLOCK_SIZE = 2                  # 경계 양쪽에서 비교할 청크 수
MIN_GAP_DISTANCE = 1            # 후보 간 최소 간격 (청크 경계 단위, 1이면 제한 없음)
EMBED_BATCH_SIZE = 32
DEVICE = None                   # None: 자동 (GPU 있으면 cuda)

# ── 평가 기준 ─────────────────────────────────────────
MATCH_TOLERANCE = 8             # 정답 경계 ±N발화 안에 후보가 있으면 '포함' (순위 기준)
REPORT_TOLERANCES = [0, 3, 8, 10]   # 참고용으로 함께 볼 허용 오차 (0 = 위치 완전 일치)

In [ ]:
# --------------------------------------------------------------------------------------------------
# 1-1. 전사문 / 정답 경계 불러오기
# 정답 boundary_positions(1부터) → 0부터 세는 '새 구간 첫 발화 번호'로 변환
# --------------------------------------------------------------------------------------------------
import json
import math
import numpy as np
import pandas as pd
from IPython.display import display

# 전사문 읽기 - 머리말, 발화 분리
def load_transcript(path, header_lines):
    lines = path.read_text(encoding="utf-8-sig").splitlines()
    header = {}
    for line in lines[:header_lines]:
        key, _, value = line.partition(":")
        header[key.strip()] = value.strip()
    utterances = [line.strip() for line in lines[header_lines:] if line.strip()]
    return header, utterances


def load_gold(path):
    g = json.loads(path.read_text(encoding="utf-8"))
    return g["utterance_count"], sorted(p - 1 for p in g["boundary_positions"])


meeting_ids = MEETING_IDS or sorted(p.stem for p in GOLD_DIR.glob("*.json"))
meetings = {}
for mid in meeting_ids:
    header, utts = load_transcript(TRANSCRIPT_DIR / f"{mid}.txt", HEADER_LINES)
    n_gold, gold = load_gold(GOLD_DIR / f"{mid}.json")
    assert n_gold == len(utts), f"{mid}: 정답 발화 수({n_gold})와 전사문 발화 수({len(utts)})가 다릅니다."
    meetings[mid] = {"header": header, "utterances": utts, "utt_chars": np.array([len(u) for u in utts]),
                     "gold": gold}

meetings_df = pd.DataFrame([{"meeting_id": mid, "utterances": len(m["utterances"]),
                             "chars": int(m["utt_chars"].sum()), "gold_boundaries": len(m["gold"])}
                            for mid, m in meetings.items()])
display(meetings_df)
print(f"회의 {len(meetings)}개 / 발화 {meetings_df['utterances'].sum():,}개 / 정답 경계 {meetings_df['gold_boundaries'].sum()}개")

In [ ]:
# 전사문 (머리말, 발화 분리된 구조)
meetings

In [ ]:
# --------------------------------------------------------------------------------------------------
# 1-2. 글자 수 기준 발화 단위 청킹
# 01 노트북과 같은 방식 (발화 중간에서 자르지 않음), 길이 단위만 토큰 → 글자 수
# --------------------------------------------------------------------------------------------------
def count_chars(text):
    # 공백·줄바꿈·문장부호 포함 글자 수
    return len(text)


def chunk_utterances(utterances, chunk_size, chunk_overlap, sep=UTTERANCE_SEPARATOR):
    chunks = []
    start = 0
    while start < len(utterances):
        end = start + 1
        while end < len(utterances) and count_chars(sep.join(utterances[start:end + 1])) <= chunk_size:
            end += 1
        text = sep.join(utterances[start:end])
        chunks.append({"text": text,
                       "metadata": {"chunk_index": len(chunks), "char_count": count_chars(text),
                                    "utterance_start": start, "utterance_end": end - 1}})
        if end == len(utterances):
            break
        next_start = end
        while next_start - 1 > start and count_chars(sep.join(utterances[next_start - 1:end])) <= chunk_overlap:
            next_start -= 1
        start = next_start
    return chunks


# 청킹은 임베딩 모델과 무관 → (회의, 청크 크기, 오버랩)별로 한 번만 계산
chunk_cache = {}
for mid, m in meetings.items():
    for size in CHUNK_SIZES:
        for overlap in OVERLAP_SIZES:
            assert overlap < size, "오버랩은 청크 크기보다 작아야 합니다."
            chunk_cache[(mid, size, overlap)] = chunk_utterances(m["utterances"], size, overlap)

chunk_stats = pd.DataFrame([{"chunk_size": s, "overlap": o, "meeting_id": mid, "num_chunks": len(cs)}
                            for (mid, s, o), cs in chunk_cache.items()])
display(chunk_stats.groupby(["chunk_size", "overlap"])["num_chunks"].agg(["sum", "mean", "min"]).round(1))

In [ ]:
# --------------------------------------------------------------------------------------------------
# 1-3. 모델별 발화 임베딩 (문장 단위 캐시)
# - 같은 문장("네.", "음." 등)은 회의가 달라도 한 번만 임베딩
# - EMBED_CACHE_DIR/<모델>/texts.json (문장 목록) + vectors.npy (같은 순서의 벡터)로 저장
# - 실행할 때마다 캐시에 없는 '처음 보는 문장'만 계산해서 캐시에 추가
#   → 두 번째 실행부터는 모델을 불러오지도 않음 / 회의가 추가되면 새 문장만 계산
# - 모델은 하나씩 불러오고 다 쓰면 메모리에서 내림
# --------------------------------------------------------------------------------------------------
import gc
import time
from sentence_transformers import SentenceTransformer


# 정규화
def l2norm(v):
    return v / np.linalg.norm(v, axis=-1, keepdims=True)


def load_text_cache(cache_dir):
    texts_path, vec_path = cache_dir / "texts.json", cache_dir / "vectors.npy"
    if not (texts_path.is_file() and vec_path.is_file()):
        return [], None
    texts, vectors = json.loads(texts_path.read_text(encoding="utf-8")), np.load(vec_path)
    assert len(texts) == len(vectors), f"{cache_dir}: 캐시 문장 수와 벡터 수가 다릅니다. 폴더를 지우고 다시 실행하세요."
    return texts, vectors


def save_text_cache(cache_dir, texts, vectors):
    cache_dir.mkdir(parents=True, exist_ok=True)
    np.save(cache_dir / "vectors.npy", vectors.astype(np.float32))
    (cache_dir / "texts.json").write_text(json.dumps(texts, ensure_ascii=False), encoding="utf-8")


all_utterances = [u for m in meetings.values() for u in m["utterances"]] # 발화문 텍스트
unique_utterances = list(dict.fromkeys(all_utterances))       # 순서 유지 중복 제거
print(f"전체 발화 {len(all_utterances):,}개 → 서로 다른 문장 {len(unique_utterances):,}개 "
      f"(중복 {1 - len(unique_utterances) / len(all_utterances):.1%} 절약)\n")

utt_embeddings = {}     # (모델 이름, 회의ID) → (발화 수, 차원)
for name, (model_id, load_kwargs, encode_kwargs) in EMBEDDING_MODELS.items():
    started = time.perf_counter()
    cache_dir = EMBED_CACHE_DIR / name
    cached_texts, cached_vectors = load_text_cache(cache_dir)
    known = set(cached_texts)
    new_texts = [t for t in unique_utterances if t not in known]
    if new_texts:
        model = SentenceTransformer(model_id, device=DEVICE, **load_kwargs)
        new_vectors = model.encode(new_texts, batch_size=EMBED_BATCH_SIZE, convert_to_numpy=True,
                                   normalize_embeddings=True, show_progress_bar=True, **encode_kwargs)
        cached_texts = cached_texts + new_texts
        cached_vectors = new_vectors if cached_vectors is None else np.vstack([cached_vectors, new_vectors])
        save_text_cache(cache_dir, cached_texts, cached_vectors)
        del model
        gc.collect()

    lookup = {t: i for i, t in enumerate(cached_texts)}
    vectors = l2norm(cached_vectors.astype(np.float32))
    for mid, m in meetings.items():
        utt_embeddings[(name, mid)] = vectors[[lookup[u] for u in m["utterances"]]]
    print(f"[{name}] 캐시 재사용 {len(unique_utterances) - len(new_texts):,}문장 / 새로 임베딩 {len(new_texts):,}문장 / "
          f"{time.perf_counter() - started:.0f}초 / 차원 {vectors.shape[1]}")

In [ ]:
# --------------------------------------------------------------------------------------------------
# 1-4. 경계 후보 탐지 함수 (01 노트북과 동일, 길이 단위만 글자 수)
# --------------------------------------------------------------------------------------------------
# 청크마다 청크 속 한 발화들의 임베딩을 토큰 수로 가중 평균하여, 청크 하나당 벡터 하나를 만드는 함수
def pooled_chunk_embeddings(chunks, utt_emb, utt_chars, filler_max_chars):
    vectors = []
    for c in chunks:
        m = c["metadata"]
        idx = np.arange(m["utterance_start"], m["utterance_end"] + 1)
        content = idx[utt_chars[idx] > filler_max_chars]
        idx = content if len(content) else idx
        weights = utt_chars[idx][:, None]                            # 가중치 = 발화 글자 수
        vectors.append((utt_emb[idx] * weights).sum(axis=0))
    return l2norm(np.array(vectors))

# 청크별(gap) 유사도 측정
# -> gap 마다 왼쪽 gap과 오른쪽 gap의 코사인 유사도를 계산해 gap 개수만큼의 유사도 배열 돌려줌
def block_similarities(emb, block_size):
    sims = []
    for g in range(len(emb) - 1):
        left = l2norm(emb[max(0, g - block_size + 1):g + 1].mean(axis=0))
        right = l2norm(emb[g + 1:g + 1 + block_size].mean(axis=0))
        sims.append(float(left @ right))
    return np.array(sims)

# 각 지점에서 왼쪽/오른쪽으로 유사도가 오르는 동안 따라가 찾은 봉우리 높이와의 차이 합 계산
# -> 해당 값(depth)으로 추후 경계 후보 선정에 활용 (depth >= 임계값 and depth > 0인 gap을 경계 후보로 정할 때)
def depth_scores(sims):
    depths = []
    for i in range(len(sims)):
        j, left_peak = i, sims[i]
        while j > 0 and sims[j - 1] >= left_peak:
            j -= 1
            left_peak = sims[j]
        j, right_peak = i, sims[i]
        while j < len(sims) - 1 and sims[j + 1] >= right_peak:
            j += 1
            right_peak = sims[j]
        depths.append((left_peak - sims[i]) + (right_peak - sims[i]))
    return np.array(depths)


def pick_by_order(depths, max_count, min_gap_distance, threshold=None):
    # depth 큰 순서로 고르면서 (임계값 미달 시 중단) 가까운 후보는 건너뜀
    chosen = []
    for g in np.argsort(-depths):
        if max_count is not None and len(chosen) >= max_count:
            break
        if threshold is not None and (depths[g] < threshold or depths[g] <= 0):
            break
        if all(abs(g - c) >= min_gap_distance for c in chosen):
            chosen.append(int(g))
    return sorted(chosen)


# gap을 후보 선택
def select_candidates(depths, method, params, min_gap_distance=MIN_GAP_DISTANCE):
    if len(depths) == 0:
        return []
    if method == "threshold":      # 1) depth > 0 이고 depth ≥ 평균 + K × 표준편차
        threshold = depths.mean() + params["std_k"] * depths.std()
        return pick_by_order(depths, None, min_gap_distance, threshold=threshold)
    if method == "ratio":          # 2) depth 상위 N% (전체 청크 경계 수 기준)
        return pick_by_order(depths, max(1, math.ceil(len(depths) * params["ratio"])), min_gap_distance)
    raise ValueError(f"알 수 없는 선택 방식: {method}")


# 뽑힌 gap을 몇 번 발화 위치로 볼 것인지 정함
# -> 즉, '몇 번 발화부터 새 주제인가?'라는 발화 단위 위치가 필요하기 때문에 해당 함수를 통해 변환
# ex) chunk 19 : 319:337
#     chunk 20 : 335:359
#     -> overlap :335 336 337 -> 336을 발화 경계 구간으로 (오버랩 구간의 가운데 발화 번호 반환)
def boundary_position(chunks, g):
    # 두 청크의 오버랩 발화 구간 가운데 = 새 구간 첫 발화 (0부터 세는 번호)
    right_start = chunks[g + 1]["metadata"]["utterance_start"]
    left_end = chunks[g]["metadata"]["utterance_end"] + 1
    return (right_start + left_end) // 2


def covered(gold, pred, tol):
    # 정답 경계마다 ±tol 안에 후보가 있는지 (True/False 목록)
    return [any(abs(p - b) <= tol for p in pred) for b in gold]

## 2. 조합별 경계 후보 추출 및 정답 포함률

| 열 | 의미 |
|---|---|
| `recall@N` | 정답 경계 중 ±N발화 안에 후보가 있는 비율 (`recall@0` = 위치 완전 일치) |
| `num_candidates` | 회의 20개에서 뽑힌 후보 수 합계 |
| `candidate_ratio` | 후보 수 / 전체 청크 경계 수 (얼마나 많이 뽑았는지) |
| `cand_per_gold` | 후보 수 / 정답 경계 수 (1에 가까울수록 적은 후보로 정답을 잡음) |
| `min_meeting_recall` | 회의별 포함률 중 가장 낮은 값 (특정 회의에서 크게 놓치는지 확인) |

In [ ]:
# --------------------------------------------------------------------------------------------------
# 2-1. 조합별 후보 추출 + 정답 포함률 계산
# 결과: candidates_store[(모델, 청크, 오버랩, 선택방식)][회의ID] = 후보 위치 목록 (0부터 세는 발화 번호)
# 순위: recall@MATCH_TOLERANCE 높은 순 → 같으면 후보 수 적은 순
# --------------------------------------------------------------------------------------------------
from itertools import product

candidates_store = {}
rows = []
total_gold = sum(len(m["gold"]) for m in meetings.values())
for name, size, overlap in product(EMBEDDING_MODELS, CHUNK_SIZES, OVERLAP_SIZES):
    depth_cache = {}
    for mid, m in meetings.items():
        chunks = chunk_cache[(mid, size, overlap)]
        emb = pooled_chunk_embeddings(chunks, utt_embeddings[(name, mid)], m["utt_chars"], FILLER_MAX_CHARS)
        depths = depth_scores(block_similarities(emb, BLOCK_SIZE))
        depth_cache[mid] = (depths, [boundary_position(chunks, g) for g in range(len(depths))])

    for method, params in SELECTION_METHODS.items():
        key = (name, size, overlap, method)
        candidates_store[key] = {}
        hits = {tol: 0 for tol in REPORT_TOLERANCES + [MATCH_TOLERANCE]}
        n_cand = n_gaps = 0
        meeting_recalls = []
        for mid, (depths, positions) in depth_cache.items():
            gold = meetings[mid]["gold"]
            pred = [positions[g] for g in select_candidates(depths, method, params)]
            candidates_store[key][mid] = pred
            for tol in hits:
                hits[tol] += sum(covered(gold, pred, tol))
            meeting_recalls.append(np.mean(covered(gold, pred, MATCH_TOLERANCE)) if gold else 1.0)
            n_cand += len(pred)
            n_gaps += len(depths)
        rows.append({"model": name, "chunk_size": size, "overlap": overlap, "selection": method,
                     **{f"recall@{tol}": hits[tol] / total_gold for tol in sorted(set(hits))},
                     "num_candidates": n_cand, "candidate_ratio": n_cand / n_gaps,
                     "cand_per_gold": n_cand / total_gold, "min_meeting_recall": min(meeting_recalls)})

rank_col = f"recall@{MATCH_TOLERANCE}"
result_df = (pd.DataFrame(rows).sort_values([rank_col, "num_candidates"], ascending=[False, True])
             .reset_index(drop=True))
result_df.index += 1
result_df.index.name = "rank"
print(f"조합 {len(result_df)}개 / 정답 경계 {total_gold}개 / 순위 기준: {rank_col} (같으면 후보 수 적은 순)")
display(result_df.head(20).round(3))

In [ ]:
# --------------------------------------------------------------------------------------------------
# 2-2. 요소별 비교 (각 요소 값별 평균)
# 한 요소의 효과를 볼 때는 나머지 요소들의 평균으로 비교
# --------------------------------------------------------------------------------------------------
metric_cols = [f"recall@{tol}" for tol in sorted(set(REPORT_TOLERANCES + [MATCH_TOLERANCE]))] + ["num_candidates", "cand_per_gold"]
for col in ["model", "chunk_size", "overlap", "selection"]:
    display(result_df.groupby(col)[metric_cols].mean().sort_values(rank_col, ascending=False).round(3))

print(f"모델 × 청크 크기별 {rank_col} (오버랩·선택 방식 평균)")
display(result_df.pivot_table(index="model", columns="chunk_size", values=rank_col, aggfunc="mean").round(3))
print("모델 × 선택 방식별 평균 후보 수")
display(result_df.pivot_table(index="model", columns="selection", values="num_candidates", aggfunc="mean").round(0))

In [ ]:
# --------------------------------------------------------------------------------------------------
# 2-3. 최적 조합 상세: 회의별 포함률과 놓친 정답 경계
# BEST_COMBO = None이면 2-1의 1위 조합, 직접 보려면 예: ("bge-m3", 400, 50, "threshold")
# 놓친 경계는 GoldenSet과 같은 1부터 세는 번호로 표시
# --------------------------------------------------------------------------------------------------
BEST_COMBO = None

best = BEST_COMBO or tuple(result_df.iloc[0][["model", "chunk_size", "overlap", "selection"]])
print("조합:", best)
detail = []
for mid, pred in candidates_store[best].items():
    gold = meetings[mid]["gold"]
    cov = covered(gold, pred, MATCH_TOLERANCE)
    detail.append({"meeting_id": mid, "gold": len(gold), "candidates": len(pred),
                   "covered": sum(cov), "recall": np.mean(cov) if gold else 1.0,
                   "missed_gold (1부터)": [b + 1 for b, c in zip(gold, cov) if not c]})
detail_df = pd.DataFrame(detail).sort_values("recall")
display(detail_df.round(3))
print(f"전체 포함률 {detail_df['covered'].sum()}/{detail_df['gold'].sum()} = "
      f"{detail_df['covered'].sum() / detail_df['gold'].sum():.3f} / 후보 {detail_df['candidates'].sum()}개")

In [ ]:
# --------------------------------------------------------------------------------------------------
# 2-4. 결과 저장
# 01 노트북과 같이 기본은 저장하지 않음. 저장하려면 주석 풀고 실행
# - 조합별 결과표 (CSV)
# - 최적 조합의 회의별 후보 위치 (JSON, GoldenSet과 같은 1부터 세는 번호) → 다음 단계(LLM 검증) 입력으로 사용
# (임베딩 캐시는 이 셀과 관계없이 자동 저장됨)
# --------------------------------------------------------------------------------------------------
EXP_DIR.mkdir(parents=True, exist_ok=True)
result_df.to_csv(EXP_DIR / "candidate_combinations.csv", encoding="utf-8-sig")
best_candidates = {
    "combo": dict(zip(["model", "chunk_size", "overlap", "selection"], best)),
    "settings": {"filler_max_chars": FILLER_MAX_CHARS, "block_size": BLOCK_SIZE,
                 "min_gap_distance": MIN_GAP_DISTANCE, "selection_params": SELECTION_METHODS[best[3]],
                 "position_definition": "1부터 세는 새 구간 첫 발화 번호 (GoldenSet과 동일)"},
    "candidates": {mid: [p + 1 for p in pred] for mid, pred in candidates_store[best].items()},
}
(EXP_DIR / "best_candidates.json").write_text(json.dumps(best_candidates, ensure_ascii=False, indent=2,
                                                         default=int), encoding="utf-8")
print("저장 완료:", EXP_DIR)